# Euler Diagrams with Rectangles — MILP

**Variables**
- Element positions `(ex_i, ey_i)` — movable small squares
- Set rectangles `(rx_j, ry_j, rw_j, rh_j)` — bottom-left corner + width/height

**Constraints** (from a boolean membership matrix)
- *Containment*: element `i` ∈ set `j` → 4 linear inequalities, no binaries
- *Exclusion*: element `i` ∉ set `j` → disjunctive (OR of 4 separating planes), 4 binaries per pair

**Objective**: minimize total perimeter `∑ 2(rw_j + rh_j)`

In [ ]:
import numpy as np
import pulp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgba

In [ ]:
from vizopt.milp_euler_rectangles import solve_euler_rectangles

In [ ]:
def plot_euler(membership, result, r=0.15, set_labels=None, element_labels=None, colors=None):
    N, S = membership.shape
    if set_labels is None:
        set_labels = [f"Set {j}" for j in range(S)]
    if element_labels is None:
        element_labels = [str(i) for i in range(N)]
    if colors is None:
        colors = plt.cm.tab10(np.linspace(0, 0.9, S))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal")

    areas = result["rectangles"][:, 2] * result["rectangles"][:, 3]
    for j in np.argsort(-areas):
        x, y, w, h = result["rectangles"][j]
        ax.add_patch(patches.Rectangle(
            (x, y), w, h, linewidth=2,
            edgecolor=colors[j], facecolor=to_rgba(colors[j], 0.12), zorder=2
        ))
        ax.text(x + w / 2, y + h + 0.05, set_labels[j],
                ha="center", va="bottom", color=colors[j], fontsize=11, fontweight="bold")

    for i in range(N):
        ex, ey = result["element_positions"][i]
        ax.add_patch(patches.Rectangle(
            (ex - r, ey - r), 2 * r, 2 * r,
            linewidth=1, edgecolor="black", facecolor="#555", zorder=3
        ))
        ax.text(ex, ey, element_labels[i], ha="center", va="center",
                fontsize=7, color="white", zorder=4)

    rects = result["rectangles"]
    all_x = np.concatenate([rects[:, 0], rects[:, 0] + rects[:, 2], result["element_positions"][:, 0]])
    all_y = np.concatenate([rects[:, 1], rects[:, 1] + rects[:, 3], result["element_positions"][:, 1]])
    pad = 0.4
    ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
    ax.set_ylim(all_y.min() - pad, all_y.max() + pad)

    ax.set_title(f"status: {result['status']}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## Example 1 — two disjoint sets

In [ ]:
membership = np.array([
    [1, 0],
    [1, 0],
    [1, 0],
    [0, 1],
    [0, 1],
    [0, 1],
], dtype=bool)

result = solve_euler_rectangles(membership)
plot_euler(membership, result, set_labels=["A", "B"])

## Example 2 — nested sets (A ⊆ B)

In [ ]:
membership = np.array([
    [1, 1],  # ∈ A and B
    [1, 1],
    [0, 1],  # ∈ B only
    [0, 1],
    [0, 1],
], dtype=bool)

result = solve_euler_rectangles(membership, offset=[0.1, 0.2])
plot_euler(membership, result, set_labels=["A", "B"])

## Example 3 — A and B disjoint, both contained in C

In [ ]:
membership = np.array([
    [1, 0, 1],  # ∈ A, C
    [1, 0, 1],
    [1, 0, 1],
    [0, 1, 1],  # ∈ B, C
    [0, 1, 1],
    [0, 1, 1],
    [0, 0, 1],  # ∈ C only
], dtype=bool)

result = solve_euler_rectangles(membership, offset=[0.1, 0.1, 0.2])
plot_euler(membership, result, set_labels=["A", "B", "C"])

## Example 4 - A liar diagram

The diagram lies: the three rectangles create a visible triple-overlap zone in the center, implying A∩B∩C ≠ ∅ — even though no element belongs to all three sets.


In [ ]:
membership = np.array([
    [1, 1, 0],  # ∈ A, B
    [0, 1, 1],  # ∈ B, C
    [1, 0, 1],  # ∈ A, C
], dtype=bool)

result = solve_euler_rectangles(membership, offset=[0.1, 0.2, 0.3])

In [ ]:
plot_euler(membership, result, set_labels=["A", "B", "C", "D"])

## Really infeasible: an octahedron

In [ ]:
membership = np.array([
    [1, 0, 1, 0, 1, 0],  # e1 ∈ {S1, S3, S5}
    [1, 0, 0, 1, 0, 1],  # e2 ∈ {S1, S4, S6}
    [0, 1, 1, 0, 0, 1],  # e3 ∈ {S2, S3, S6}
    [0, 1, 0, 1, 1, 0],  # e4 ∈ {S2, S4, S5}
], dtype=bool)

result = solve_euler_rectangles(membership)
result

## Example 5 — multiples of 2, 3, 5 among integers 2–12

In [ ]:
elements = list(range(2, 10))   # 2, 3, ..., 9
divisors = [2, 3, 5]

membership = np.array(
    [[n % d == 0 for d in divisors] for n in elements],
    dtype=bool,
)

result = solve_euler_rectangles(membership, position_penalty=1.0)
plot_euler(
    membership, result,
    set_labels=["×2", "×3", "×5"],
    element_labels=[str(n) for n in elements],
)

In [ ]:
elements = list(range(2, 12))   # 2, 3, ..., 9
divisors = [2, 3, 5]

membership = np.array(
    [[n % d == 0 for d in divisors] for n in elements],
    dtype=bool,
)

result = solve_euler_rectangles(membership, position_penalty=1.0)
plot_euler(
    membership, result,
    set_labels=["×2", "×3", "×5"],
    element_labels=[str(n) for n in elements],
)

In [ ]:
2